In [1]:
from langchain_core.documents import Document

In [11]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
documents =  [
    Document(page_content="Hello, My Name is Binsha .I am an AI Engineer", metadata={"source": "file1.txt"}),
    Document(page_content="My name is Gourangi", metadata={"source": "file2.txt"}),
    Document(page_content="I am a Data Scientist", metadata={"source": "file3.txt"}),
    Document(page_content="I am a Software Engineer", metadata={"source": "file4.txt"})


]

In [13]:
groq_api_key = os.environ.get("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN")

In [46]:
from langchain_groq import ChatGroq
llm  = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.7
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020D623047D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020D62305090>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [47]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1436.43it/s]


In [48]:
from langchain_chroma import Chroma

In [49]:
vector_store = Chroma.from_documents(documents, embeddings)
vector_store

In [50]:
vector_store.similarity_search("What is your name?", k=2)

[Document(id='7788c83f-9d7e-4850-a12b-b4c5135e9529', metadata={'source': 'file2.txt'}, page_content='My name is Gourangi'),
 Document(id='ee09c333-0858-4c87-a3e8-bb420f058d3e', metadata={'source': 'file2.txt'}, page_content='My name is Gourangi')]

In [51]:
vector_store.similarity_search_with_score("What is your name?", k=2)

[(Document(id='7788c83f-9d7e-4850-a12b-b4c5135e9529', metadata={'source': 'file2.txt'}, page_content='My name is Gourangi'),
  0.8882598876953125),
 (Document(id='ee09c333-0858-4c87-a3e8-bb420f058d3e', metadata={'source': 'file2.txt'}, page_content='My name is Gourangi'),
  0.8882598876953125)]

##Retrievers ###

In [52]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda


In [53]:
retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(["data engineer","dog"])

[[Document(id='b058b626-d6ce-4e36-9cc0-050f65be8242', metadata={'source': 'file3.txt'}, page_content='I am a Data Scientist')],
 [Document(id='7788c83f-9d7e-4850-a12b-b4c5135e9529', metadata={'source': 'file2.txt'}, page_content='My name is Gourangi')]]

In [54]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 1})   
retriever.batch(["data engineer","dog"])

[[Document(id='b058b626-d6ce-4e36-9cc0-050f65be8242', metadata={'source': 'file3.txt'}, page_content='I am a Data Scientist')],
 [Document(id='7788c83f-9d7e-4850-a12b-b4c5135e9529', metadata={'source': 'file2.txt'}, page_content='My name is Gourangi')]]

In [55]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer the question using provided context, if the answer is not contained within the text below, say "I don't know".
{question}
context:
{context}

"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain ={"context": retriever, "question": RunnablePassthrough() } | prompt | llm





In [56]:
response = rag_chain.invoke("What is your name?")

In [58]:
response.content

'Gourangi'